# Gait Feature Extraction & XGBoost Pipeline

Cleaned notebook: end-to-end pipeline from raw pose data to tabular gait features and a first XGBoost model.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

import gait_preprocessing_pipeline as gait
import feature_extraction as fx

## 1. Load raw pose data

Adjust `db_path` and the SQL query as needed. This cell loads the long-format MediaPipe landmarks into `df` and creates a copy `Test_data`.

In [ ]:
import sqlite3

# TODO: set your actual database path if different
db_path = r"C:/path/to/landmark_database.db"  # adjust as needed
conn = sqlite3.connect(db_path)

# Example query – adapt WHERE / LIMIT for your use case
df = pd.read_sql_query("SELECT * FROM landmarks", conn)
Test_data = df.copy()
df.head()

## 2. Build video-level dataframe (`df_video`)

Convert long-format landmarks into one row per video with a `pose` tensor.

In [ ]:
df_video = gait.add_pose_column(Test_data)
df_video.head()

### Sanity checks

- Check pose tensor shape
- Inspect label / meta columns

In [ ]:
df_video.iloc[0]['pose'].shape, df_video.columns

## 3. Extract gait features

Use the shared `feature_extraction.py` module to compute tabular features per video, including knee motion and symmetry metrics.

In [ ]:
df_features = fx.extract_features_from_df_video(df_video)
df_features.head()

### Label distribution

Inspect how many samples we have per 5-class label.

In [ ]:
df_features['label_class'].value_counts(dropna=False)

## 4. XGBoost 5-class baseline

Train a simple multi-class XGBoost model on the extracted features.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Drop non-numeric/meta columns
drop_cols = ['label_fine', 'label_class', 'label_id',
             'movement_type', 'side', 'source_file']
drop_cols = [c for c in drop_cols if c in df_features.columns]

X = df_features.drop(columns=drop_cols)
y = df_features['label_id'].astype('int64')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(fx.CLASS_MAP),
    eval_metric='mlogloss',
    tree_method='hist',
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))